# HDB Resale Price Index Cleaning

This notebook extracts the quarterly HDB Resale Price Index (RPI) from the published PDF table and reshapes it into a tidy CSV.

## Cleaning objective
- parse quarterly index values from the raw HDB PDF
- standardize the period field into `year` and `quarter`
- remove duplicate period entries
- export the cleaned dataset to the project `data/cleaned` folder


In [ ]:
# ============================================================
# 1. Imports and configuration
# ============================================================

from pathlib import Path
import re

import pandas as pd
import pdfplumber

INPUT_FILE = Path("../../data/raw/4Q2025 RPI Table.pdf")
OUTPUT_FILE = Path("../../data/clean/rpi_cleaned.csv")


## Extract quarterly index values from the PDF

The helper function scans the full PDF text, identifies yearly sections, matches quarter-index pairs, and returns one standardized row per period.


In [2]:
# ============================================================
# 2. Helper function
# ============================================================

def extract_hdb_rpi(file_path: Path) -> pd.DataFrame:
    """Extract quarterly HDB resale price index values from the source PDF."""
    all_text = ""
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text() or ""
            all_text += page_text + "\n"

    years = re.findall(r"\b(199\d|20[0-2]\d)\b", all_text)
    years = sorted(set(years), reverse=True)

    quarter_map = {
        "1": "Q1",
        "I": "Q1",
        "11": "Q2",
        "II": "Q2",
        "III": "Q3",
        "IV": "Q4",
    }
    records = []

    for index, year in enumerate(years):
        next_year = years[index + 1] if index + 1 < len(years) else "$"
        section_match = re.search(f"{year}(.*?)(?={next_year})", all_text, re.DOTALL)
        if not section_match:
            continue

        section_text = section_match.group(1)
        matches = re.findall(r"\b(IV|III|II|I|11|1)\b\s+(\d{2,3}\.\d)\b", section_text)

        for quarter_raw, index_value in matches:
            records.append(
                {
                    "Period": f"{year}-{quarter_map[quarter_raw]}",
                    "Index": float(index_value),
                }
            )

    return (
        pd.DataFrame(records)
        .drop_duplicates(subset=["Period"])
        .sort_values("Period", ascending=False)
        .reset_index(drop=True)
    )


## Reshape and export the cleaned series

This section converts the period field into tidy year and quarter columns, previews the result, and saves the cleaned table.


In [3]:
# ============================================================
# 3. Build cleaned dataset
# ============================================================

df_rpi = extract_hdb_rpi(INPUT_FILE)
df_rpi["year"] = df_rpi["Period"].str.split("-").str[0]
df_rpi["quarter"] = df_rpi["Period"].str.split("-").str[1].str.replace("Q", "", regex=False)
df_rpi = df_rpi.drop(columns=["Period"])

print("RPI records:", len(df_rpi))
df_rpi.head()


RPI records: 144


,Index,year,quarter
0,203.6,2025,4
1,203.7,2025,3
2,202.9,2025,2
3,201.0,2025,1
4,180.4,2024,4


In [4]:
# ============================================================
# 4. Save cleaned dataset
# ============================================================

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
df_rpi.to_csv(OUTPUT_FILE, index=False)
print(f"Saved cleaned RPI data to {OUTPUT_FILE.resolve()}")


Saved cleaned RPI data to /Users/celine/Documents/GitHub/DSA4264-Public-Policy-and-Society/data/cleaned/rpi_cleaned.csv
